# Parquet 转 MySQL 导入 CSV

将四张清洗后的 Parquet 表导出为 UTF-8 CSV，供 MySQL `LOAD DATA LOCAL INFILE` 使用。


In [1]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"


In [2]:
MYSQL_IMPORT_DIR.mkdir(parents=True, exist_ok=True)

export_map = {
    "log_standard_clean.parquet": "fact_standard_behavior.csv",
    "log_random_clean.parquet": "fact_random_behavior.csv",
    "user_features_clean.parquet": "dim_user.csv",
    "video_features_basic_clean.parquet": "dim_video.csv",
}

for parquet_name, csv_name in export_map.items():
    source_path = PROCESSED_DIR / parquet_name
    target_path = MYSQL_IMPORT_DIR / csv_name
    table = pd.read_parquet(source_path)

    if "record_id" in table.columns:
        table = table.drop(columns="record_id")

    table.to_csv(target_path, index=False, encoding="utf-8", lineterminator="\n")
    print(f"{csv_name}: {table.shape[0]:,} 行 × {table.shape[1]} 列")


fact_standard_behavior.csv: 1,414,622 行 × 25 列
fact_random_behavior.csv: 1,186,049 行 × 24 列
dim_user.csv: 27,285 行 × 32 列
dim_video.csv: 7,583 行 × 17 列
